# Two-tower model — standalone training notebook

Self-contained: **no `import two_tower`**. All library code is inlined below in clearly labelled cells.

## How to use
1. **Edit `TRAIN_CONFIG_YAML`** (cell 3) — update S3 paths, feature lists, hyperparameters.
2. **Run All** (`Kernel → Restart & Run All`).

## Notebook structure
| Cell | What it does |
|------|-------------|
| 1 | Install dependencies |
| 2–3 | Pipeline config (YAML string) |
| 4 | Shared imports |
| 5 | Config dataclasses |
| 6 | I/O utilities (S3, RunLog, MLflow) |
| 7 | Vocabulary helpers |
| 8 | Feature schema helpers |
| 9 | Hash-based embedding init |
| 10 | Feature encoding (cats, nums, multi) |
| 11 | Feature preparation |
| 12 | Config loader |
| 13 | YAML string loader (notebook helper) |
| 14 | Data loading |
| 15 | Dataset class |
| 16 | Model architecture (DCNv2 + MLP towers) |
| 17 | Training helpers |
| 18 | `train_and_log()` |
| 19 | Parse config |
| 20 | Optional per-client downsampling |
| 21 | Load data → prepare features → **train** |


In [ ]:
import subprocess
import sys
from pathlib import Path


def _find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for p in [cwd, *cwd.parents]:
        if (p / "requirements-mlops.txt").exists():
            return p
        if (p / "configs" / "train.yaml").exists():
            return p
    return cwd


_REPO = _find_repo_root()
_REQ = _REPO / "requirements-mlops.txt"
if _REQ.is_file():
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_REQ)])
else:
    subprocess.check_call(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "torch>=2.2.0", "pandas>=2.0.0", "numpy>=1.26.0",
            "scikit-learn>=1.4.0", "mlflow>=2.10.0",
            "pyarrow>=15.0.0", "s3fs>=2024.1.0",
            "polars>=0.20.0", "PyYAML>=6.0.0",
        ]
    )
print("Dependencies OK.")


## Config
Edit the YAML below — same schema as `configs/train.yaml`.

Key sections:
- **`paths`**: S3 (or local) locations for train/val/infer data and artifact output.
- **`features`**: column names for labels, device-id, client-id, user features, client features.
- **`train`**: hyperparameters (batch size, epochs, learning rate, embedding dim, …).
- **`data`**: optional client-metadata join (`merge_client_metadata`).
- **`mlflow_tracking_uri`**: set to your MLflow server URL, or leave `null` for local tracking.


In [ ]:
import yaml

TRAIN_CONFIG_YAML = r"""paths:
  train: "s3://mobavenue-simplismart-aws-s3-apse-sg/rtb/data/two_tower_ml/combined_data/training_data_final/"
  val: "s3://mobavenue-simplismart-aws-s3-apse-sg/rtb/data/two_tower_ml/combined_data/validation_data_final/"
  infer: "s3://mobavenue-simplismart-aws-s3-apse-sg/rtb/data/two_tower_ml/test_data_final/20260416_full/"
  artifacts_base: "s3://mobavenue-simplismart-aws-s3-apse-sg/rtb/data/two_tower_ml/"

features:
  label_col: "label"
  device_id_col: "device_id"
  client_id_col: "client_bundle_id"
  user_feature_cols:
    - "carrier"
    - "os_version"
    - "recency"
    - "city_original"
    - "state_original"
    - "city_tier"
    - "geographic_region"
    - "is_metro"
    - "is_capital"
    - "is_coastal"
    - "literacy_rate"
    - "investments_rate"
    - "crypto_rate"
    - "finance_rate"
    - "shopping_rate"
    - "entertainment_rate"
    - "social_rate"
    - "education_rate"
    - "utility_rate"
    - "health_rate"
    - "travel_rate"
    - "news_rate"
    - "food_drink_rate"
    - "lending_rate"
    - "quick_service_rate"
    - "ride_rate"
    - "gaming_action_rate"
    - "real_money_casino_rate"
    - "word_game_rate"
    - "trivia_puzzle_rate"
    - "strategy_game_rate"
    - "sports_game_rate"
    - "simulation_rpg_rate"
    - "racing_game_rate"
    - "casual_game_rate"
    - "board_game_rate"
    - "arcade_game_rate"
    - "ad_type_0_rate"
    - "ad_type_1_rate"
    - "ad_type_2_rate"
    - "rewarded_rate"
    - "interstitial_rate"
    - "distinct_score_45plus_apps"
    - "distinct_score_40_45_apps"
    - "distinct_score_30_40_apps"
    - "distinct_score_below30_apps"
    - "distinct_mainstream_apps"
    - "distinct_popular_apps"
    - "distinct_mid_tier_apps"
    - "distinct_niche_apps"
    - "distinct_new_apps"
    - "distinct_recent_apps"
    - "distinct_established_apps"
    - "distinct_veteran_apps"
    - "distinct_free_apps"
    - "distinct_paid_apps"
    - "distinct_iap_apps"
    - "distinct_rating_everyone_apps"
    - "distinct_rating_teen_apps"
    - "distinct_rating_mature_apps"
    - "distinct_rating_everyone10plus_apps"
    - "distinct_rating_12plus_apps"
    - "distinct_rating_16plus_apps"
    - "distinct_rating_18plus_apps"
    - "distinct_rating_adults18plus_apps"
    - "bids_day_rate"
    - "bids_night_rate"
    - "bids_weekday_rate"
    - "bids_weekend_rate"
    - "bids_hour_0_2_rate"
    - "bids_hour_3_5_rate"
    - "bids_hour_6_8_rate"
    - "bids_hour_9_11_rate"
    - "bids_hour_12_14_rate"
    - "bids_hour_15_17_rate"
    - "bids_hour_18_20_rate"
    - "bids_hour_21_23_rate"
    - "total_events"
    - "bids_event_count"
    - "total_distinct_apps"
    - "unique_bid_apps"
    - "bids_apps_top_1"
    - "bids_apps_top_2"
    - "bids_apps_top_3"
    - "bids_apps_top_4"
    - "bids_apps_top_5"
    - "bids_apps_top_6"
    - "bids_apps_top_7"
    - "bids_apps_top_8"
    - "bids_apps_top_9"
    - "bids_apps_top_10"
    - "bids_genres_top_1"
    - "bids_genres_top_2"
    - "bids_genres_top_3"
    - "bids_genres_top_4"
    - "bids_genres_top_5"
    - "organic_apps_top_1"
    - "organic_apps_top_2"
    - "organic_apps_top_3"
    - "organic_apps_top_4"
    - "organic_apps_top_5"
    - "organic_apps_top_6"
    - "organic_apps_top_7"
    - "organic_apps_top_8"
    - "organic_apps_top_9"
    - "organic_apps_top_10"
    - "organic_genres_top_1"
    - "organic_genres_top_2"
    - "organic_genres_top_3"
    - "organic_genres_top_4"
    - "organic_genres_top_5"
  client_feature_cols:
    - "client_genreid"
    - "client_content_rating"
    - "client_score_45plus"
    - "client_score_40_45"
    - "client_score_30_40"
    - "client_score_below30"
    - "client_score_missing"
    - "client_installs_10m_plus"
    - "client_installs_1m_10m"
    - "client_installs_100k_1m"
    - "client_installs_below100k"
    - "client_installs_missing"
    - "client_age_0_3months"
    - "client_age_3_12months"
    - "client_age_1_2years"
    - "client_age_2plus_years"
    - "client_is_free"
    - "client_has_iap"
    - "client_rating_everyone"
    - "client_rating_teen"
    - "client_rating_mature"
    - "client_rating_everyone10plus"
    - "client_rating_rated3plus"
    - "client_rating_rated7plus"
    - "client_rating_rated12plus"
    - "client_rating_rated16plus"
    - "client_rating_rated18plus"
    - "client_rating_adults18plus"
    - "client_is_finance"
    - "client_is_shopping"
    - "client_is_food_delivery"
    - "client_is_gaming"
    - "client_is_entertainment"
    - "client_is_social"
    - "client_is_education"
    - "client_is_utility"
    - "client_is_health"
    - "client_is_travel"
    - "client_is_news"
    - "client_is_crypto"
    - "client_is_lending"
    - "client_is_investment"
    - "client_is_grocery"
  user_multi_cols: []
  client_multi_cols: []

train:
  experiment_name: "two_tower_dcnv2"
  run_name: null
  seed: 42
  batch_size: 4096
  epochs: 20
  lr: 0.001
  weight_decay: 0.0
  embed_dim: 1024
  dcn_cross_layers: 3
  mlp_hidden_dims: [128, 128]
  min_count: 1000
  num_oov_buckets: 8
  multi_max_tokens: 32
  num_workers: 4
  device: "cuda"
  pretrained_emb_dim: 128
  pretrained_cat_emb_dim: 128
  freeze_pretrained_base: true
  multi_cat_pool: "mean"
  client_mlp_hidden: [128, 128]

infer:
  topk_clients: 1
  infer_stream_batch_rows: 65536
  num_physical_gpus: 1
  workers_per_gpu: 1
  ranking_output: "s3://mobavenue-simplismart-aws-s3-apse-sg/rtb/data/two_tower_ml/user_client_rankings/"

mlflow_tracking_uri: null

# Optional client metadata:
# - merge_client_metadata: per-row join on client_id_col (multi-client train/val).
# - inject_single_client_metadata: broadcast one metadata row (single-client only; mutually exclusive).
data:
  # Set true when train/val have client_id_col but no client_* columns (join from client_metadata Parquet).
  merge_client_metadata: true
  client_metadata_uri: "s3://mobavenue-simplismart-aws-s3-apse-sg/rtb/data/two_tower_ml/client_metadata/"
  inject_single_client_metadata: false
  single_client_metadata_uri: "s3://mobavenue-simplismart-aws-s3-apse-sg/rtb/data/two_tower_ml/client_metadata/"
  single_client_row_filter: null
  single_client_features_hardcoded: null

extra: {}

"""


## Library code
The following cells define all functions and classes used by the training pipeline.
They must be run once (in order) before the training cell at the bottom.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import io
import math
import os
import pickle
import re
import socket
import time
import traceback
from collections import Counter
from contextlib import nullcontext
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable, List

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import yaml


### Config dataclasses
Frozen dataclasses that hold the parsed YAML config (paths, features, train hyper-params, etc.).


In [ ]:
# ── Config dataclasses (DataPaths, TrainConfig, PipelineConfig, …) ──────────────────────────────────────────────────────
@dataclass(frozen=True)
class DataPaths:
    train: str
    val: str
    infer: str
    artifacts_base: str  # e.g. s3://.../two_tower_ml or local path


@dataclass(frozen=True)
class FeatureConfig:
    label_col: str
    device_id_col: str
    client_id_col: str
    user_feature_cols: list[str]
    client_feature_cols: list[str]
    user_multi_cols: list[str]
    client_multi_cols: list[str]


@dataclass(frozen=True)
class TrainConfig:
    experiment_name: str
    run_name: str | None
    seed: int

    batch_size: int
    epochs: int
    lr: float
    weight_decay: float

    embed_dim: int  # final embedding dim (e.g. 1024)
    dcn_cross_layers: int
    mlp_hidden_dims: list[int]

    min_count: int
    num_oov_buckets: int
    multi_max_tokens: int

    num_workers: int
    device: str  # "cuda" or "cpu"

    # Hash embedding init for categorical columns (reference: PRETRAINED_EMB_DIM)
    pretrained_emb_dim: int = 128
    pretrained_cat_emb_dim: int = 64
    freeze_pretrained_base: bool = True

    # Multi-value categorical pooling in towers (reference: MULTI_CAT_POOL)
    multi_cat_pool: str = "mean"

    # Client MLP hidden layers (reference: CLIENT_HIDDEN); user deep stack uses mlp_hidden_dims
    client_mlp_hidden: list[int] = field(default_factory=lambda: [256, 256])

    train_eval_max_examples: int = 0


@dataclass(frozen=True)
class InferenceConfig:
    topk_clients: int
    infer_stream_batch_rows: int

    # worker pool / GPU sharding
    num_physical_gpus: int
    workers_per_gpu: int

    # outputs
    ranking_output: str

    # batching / perf (reference: RANK_USER_BATCH, CLIENT_CHUNK, etc.)
    rank_user_batch: int = 4096
    client_chunk: int = 8192
    use_amp: bool = True
    amp_dtype: str = "float16"
    output_min_rows_per_part: int = 100_000
    output_parquet_compression: str = "zstd"
    debug_cuda: bool = False

    # Optional run-size controls for quick smoke tests.
    # - max_files: only process first N parquet inputs under paths.infer
    # - max_users_per_file: stop after ranking N unique users per parquet file
    max_files: int | None = None
    max_users_per_file: int | None = None


@dataclass(frozen=True)
class InferPaths:
    """Minimal paths for a standalone inference job (`configs/infer.yaml`)."""

    infer: str
    artifacts_base: str


@dataclass(frozen=True)
class InferJobConfig:
    paths: InferPaths
    infer: InferenceConfig


@dataclass(frozen=True)
class DataLoadConfig:
    """Optional preprocessing when loading train/val (matches reference notebook)."""

    # Per-row join: ``client_metadata`` Parquet (same schema as pipeline ``client_metadata`` table)
    merge_client_metadata: bool = False
    client_metadata_uri: str | None = None

    inject_single_client_metadata: bool = False
    single_client_metadata_uri: str | None = None
    single_client_row_filter: dict[str, Any] | None = None
    single_client_features_hardcoded: dict[str, Any] | None = None


@dataclass(frozen=True)
class PipelineConfig:
    paths: DataPaths
    features: FeatureConfig
    train: TrainConfig
    infer: InferenceConfig
    data_load: DataLoadConfig = field(default_factory=DataLoadConfig)

    mlflow_tracking_uri: str | None = None
    extra: dict[str, Any] | None = None


### I/O utilities
Helpers for reading S3 / local files, structured run-logging to disk, and MLflow setup.


In [ ]:
# ── artifact_uri — join S3 or local paths ──────────────────────────────────────────────────────
def artifact_uri(base: str, *relative_parts: str) -> str:
    """Join S3 or local base with relative path parts."""
    rel = "/".join(relative_parts)
    b = base.rstrip("/")
    if b.startswith("s3://"):
        return f"{b}/{rel}"
    return str(Path(b) / rel)

# ── read_uri_bytes — read S3 or local bytes ──────────────────────────────────────────────────────
def read_uri_bytes(uri: str) -> bytes:
    uri = str(uri)
    if uri.startswith("s3://"):
        import s3fs

        with s3fs.S3FileSystem().open(uri, "rb") as f:
            return f.read()
    return Path(uri).expanduser().resolve().read_bytes()

# ── RunLog — timestamped run log file ──────────────────────────────────────────────────────
import os
import socket
import time
from dataclasses import dataclass


@dataclass
class RunLog:
    path: Path

    def write(self, msg: str) -> None:
        ts = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
        line = f"{ts} | {msg}\n"
        self.path.parent.mkdir(parents=True, exist_ok=True)
        with self.path.open("a", encoding="utf-8") as f:
            f.write(line)
            f.flush()


def _default_logs_dir() -> Path:
    # Use cwd so notebooks + CLI runs land in the same place.
    return Path(os.getcwd()).resolve() / "logs"


def start_run_log(*, kind: str, name: str | None = None, logs_dir: str | Path | None = None) -> RunLog:
    d = Path(logs_dir).resolve() if logs_dir is not None else _default_logs_dir()
    safe = (name or "run").replace(" ", "_")
    stamp = time.strftime("%Y%m%d_%H%M%S", time.localtime())
    p = d / f"{kind}_{safe}_{stamp}.log"
    rl = RunLog(path=p)
    rl.write(f"START kind={kind} name={name or ''} host={socket.gethostname()} pid={os.getpid()}")
    return rl

# ── setup_mlflow — configure tracking URI / experiment ──────────────────────────────────────────────────────
def setup_mlflow(tracking_uri: str | None, experiment_name: str) -> None:
    import mlflow

    if tracking_uri:
        mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment(experiment_name)


### Categorical vocabulary
Builds token→id mappings with OOV (out-of-vocabulary) hash buckets for categorical features.


In [ ]:
# ── Categorical vocabulary — build, encode, serialise ──────────────────────────────────────────────────────
from dataclasses import dataclass



@dataclass(frozen=True)
class CatVocab:
    token_to_id: dict[str, int]
    n_frequent: int
    num_oov_buckets: int

    @property
    def size(self) -> int:
        # 0 reserved for missing/padding
        return int(self.n_frequent + self.num_oov_buckets + 1)

    def encode_scalar(self, raw) -> int:
        if raw is None or (isinstance(raw, float) and pd.isna(raw)):
            return 0
        s = str(raw).strip()
        if s == "" or s.lower() == "nan" or s == "__NA__":
            return 0
        tid = self.token_to_id.get(s)
        if tid is not None:
            return int(tid)
        h = int(hashlib.md5(s.encode("utf-8")).hexdigest(), 16)
        return int(self.n_frequent + 1 + (h % self.num_oov_buckets))


def build_cat_vocab(series, min_count: int, num_oov_buckets: int) -> CatVocab:
    s = series.fillna("__NA__").astype(str)
    vc = s.value_counts()
    frequent = vc[vc > int(min_count)]
    tokens = list(frequent.index)
    token_to_id = {tok: i + 1 for i, tok in enumerate(tokens)}
    return CatVocab(token_to_id=token_to_id, n_frequent=len(tokens), num_oov_buckets=int(num_oov_buckets))


def parse_multi_cell(val) -> list[str]:
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    if isinstance(val, (list, tuple)):
        return [str(x).strip() for x in val if str(x).strip() and str(x).strip().lower() != "nan"]
    if isinstance(val, np.ndarray):
        return [
            str(x).strip()
            for x in val.flatten().tolist()
            if str(x).strip() and str(x).strip().lower() != "nan"
        ]
    s = str(val).strip()
    if not s or s == "__NA__" or s.lower() == "nan":
        return []
    parts = re.split(r"[\|,;]+|\s+", s)
    return [p for p in parts if p]


def build_multi_token_vocab(series, min_count: int, num_oov_buckets: int) -> CatVocab:
    ctr: Counter[str] = Counter()
    for v in series:
        for tok in parse_multi_cell(v):
            ctr[tok] += 1
    tokens = sorted([t for t, n in ctr.items() if n > int(min_count)], key=lambda t: (-ctr[t], t))
    token_to_id = {tok: i + 1 for i, tok in enumerate(tokens)}
    return CatVocab(token_to_id=token_to_id, n_frequent=len(tokens), num_oov_buckets=int(num_oov_buckets))


def vocab_to_dict(v: CatVocab) -> dict:
    return {"token_to_id": dict(v.token_to_id), "n_frequent": int(v.n_frequent), "num_oov_buckets": int(v.num_oov_buckets)}


def vocab_from_dict(d: dict) -> CatVocab:
    return CatVocab(token_to_id=dict(d["token_to_id"]), n_frequent=int(d["n_frequent"]), num_oov_buckets=int(d["num_oov_buckets"]))


### Feature schema helpers
Intersects configured feature columns with actual Parquet columns; splits into categorical vs numeric vs multi-valued.


In [ ]:
# ── Feature schema helpers — intersect columns, split cat/num/multi ──────────────────────────────────────────────────────
def intersect_feature_cols(df, cols: list[str], label: str) -> list[str]:
    """Keep only columns present in the dataframe.

    Mirrors notebook helper `_intersect_feature_cols`.
    """
    ok = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        tail = " ..." if len(missing) > 8 else ""
        print(f"{label}: skipping {len(missing)} columns not in data: {missing[:8]}{tail}")
    return ok


def split_cat_num_multi(df, cols: list[str], multi_cols: set[str]) -> tuple[list[str], list[str], list[str]]:
    """Split into scalar categorical, numeric, multi-valued categorical."""
    import pandas as pd

    cat: list[str] = []
    num: list[str] = []
    multi: list[str] = []
    for c in cols:
        if c in multi_cols:
            multi.append(c)
            continue
        dt = df[c].dtype
        if pd.api.types.is_string_dtype(dt) or pd.api.types.is_object_dtype(dt):
            cat.append(c)
        else:
            num.append(c)
    return cat, num, multi


def ensure_columns_present(df, required: Iterable[str], *, where: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        tail = " ..." if len(missing) > 10 else ""
        raise KeyError(f"{where}: missing {len(missing)} column(s): {missing[:10]}{tail}")


### Hash-based embedding initialisation
Maps each vocabulary token to a deterministic unit-normed vector via SHA-256 — used as pretrained initialisation for categorical embeddings.


In [ ]:
# ── Hash-based pretrained embedding initialisation ──────────────────────────────────────────────────────
def hash_embed_token(token: str, dim: int) -> np.ndarray:
    """Map a string to a unit-normed float32 vector (reference: SHA-256 multi-seed)."""
    v = np.empty(dim, dtype=np.float32)
    for seed in range((dim + 3) // 4):
        h = int(hashlib.sha256(f"{seed}:{token}".encode()).hexdigest(), 16)
        for i in range(4):
            idx = seed * 4 + i
            if idx < dim:
                v[idx] = ((h >> (i * 16)) & 0xFFFF) / 32768.0 - 1.0
    norm = float(np.linalg.norm(v))
    return v / norm if norm > 1e-9 else v


def build_hash_weight_matrix(vocab: CatVocab, pre_dim: int) -> np.ndarray:
    """(vocab.size × pre_dim) for nn.Embedding init (row 0 = padding; OOV rows stay zero)."""
    id_to_token = {v: k for k, v in vocab.token_to_id.items()}
    w = np.zeros((vocab.size, pre_dim), dtype=np.float32)
    for _idx, tok in id_to_token.items():
        w[_idx] = hash_embed_token(tok, pre_dim)
    return w


def build_all_hash_weights(vocabs: dict[str, CatVocab], pre_dim: int) -> dict[str, np.ndarray]:
    return {col: build_hash_weight_matrix(v, pre_dim) for col, v in vocabs.items()}


### Feature encoding
Converts a pandas DataFrame into PyTorch tensors (`encode_cats`, `encode_nums`, `encode_multi_matrix`) and defines the `collate_fn` for the DataLoader.


In [ ]:
# ── Feature encoding — encode_cats / encode_nums / encode_multi_matrix / collate_fn ──────────────────────────────────────────────────────
@dataclass(frozen=True)
class TwoTowerBatch:
    user_cat: torch.Tensor
    user_num: torch.Tensor
    user_multi: torch.Tensor
    client_cat: torch.Tensor
    client_num: torch.Tensor
    client_multi: torch.Tensor
    label: torch.Tensor


def encode_cats(df: pd.DataFrame, cat_cols: list[str], vocabs: dict) -> torch.Tensor:
    if not cat_cols:
        return torch.zeros((len(df), 0), dtype=torch.long)
    out = []
    for c in cat_cols:
        vocab = vocabs[c]
        ids = df[c].map(lambda x: vocab.encode_scalar(x)).astype("int64").to_numpy()
        out.append(torch.from_numpy(ids))
    return torch.stack(out, dim=1)


def encode_multi_matrix(
    df: pd.DataFrame, cols: list[str], vocabs: dict, max_tokens: int
) -> torch.Tensor:
    """Shape (N, len(cols), max_tokens); 0 = pad / empty slot."""
    n = len(df)
    f = len(cols)
    mt = int(max_tokens)
    if f == 0:
        return torch.zeros((n, 0, max(1, mt)), dtype=torch.long)
    out = np.zeros((n, f, mt), dtype=np.int64)
    for j, c in enumerate(cols):
        vcb = vocabs[c]
        for i, val in enumerate(df[c]):
            toks = parse_multi_cell(val)[:mt]
            for k, tok in enumerate(toks):
                tid = vcb.token_to_id.get(tok)
                if tid is not None:
                    out[i, j, k] = tid
                else:
                    h = int(hashlib.md5(tok.encode("utf-8")).hexdigest(), 16)
                    out[i, j, k] = int(vcb.n_frequent + 1 + (h % vcb.num_oov_buckets))
    return torch.from_numpy(out)


def encode_nums(df: pd.DataFrame, num_cols: list[str]) -> torch.Tensor:
    if not num_cols:
        return torch.zeros((len(df), 0), dtype=torch.float32)
    x = df[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype("float32").to_numpy()
    return torch.from_numpy(x)


def collate_fn(batch) -> TwoTowerBatch:
    uc, un, um, cc, cn, cm, y = zip(*batch)
    return TwoTowerBatch(
        user_cat=torch.stack(uc),
        user_num=torch.stack(un),
        user_multi=torch.stack(um),
        client_cat=torch.stack(cc),
        client_num=torch.stack(cn),
        client_multi=torch.stack(cm),
        label=torch.stack(y),
    )


### Feature preparation
`FeatureArtifacts` bundles vocabs and pretrained weight matrices. `prepare_training_features()` fits vocabs on train-only data and builds hash tables.


In [ ]:
# ── Feature preparation — FeatureArtifacts dataclass + prepare_training_features() ──────────────────────────────────────────────────────
@dataclass
class FeatureArtifacts:
    """Everything needed to build datasets / models after Parquet load (reference notebook)."""

    user_feature_cols: list[str]
    client_feature_cols: list[str]
    user_multi_cols: list[str]
    client_multi_cols: list[str]
    user_cat_cols: list[str]
    user_num_cols: list[str]
    client_cat_cols: list[str]
    client_num_cols: list[str]
    user_vocabs: dict[str, CatVocab]
    client_vocabs: dict[str, CatVocab]
    user_multi_vocabs: dict[str, CatVocab]
    client_multi_vocabs: dict[str, CatVocab]
    user_cat_pretrained_weights: dict[str, np.ndarray] = field(default_factory=dict)
    client_cat_pretrained_weights: dict[str, np.ndarray] = field(default_factory=dict)


def prepare_training_features(
    train_df,
    val_df,
    cfg: PipelineConfig,
    *,
    build_hash_weights: bool = True,
) -> FeatureArtifacts:
    """Intersect feature lists, split cat/num/multi, fit vocabs on train, optional hash embedding tables."""
    fc = cfg.features
    tc = cfg.train

    if fc.label_col not in train_df.columns:
        raise KeyError(
            f"Train data missing label column {fc.label_col!r}. "
            f"Columns: {list(train_df.columns)[:40]} ..."
        )

    user_cols = list(fc.user_feature_cols)
    client_cols = list(fc.client_feature_cols)
    user_cols = intersect_feature_cols(train_df, user_cols, "USER")
    user_cols = intersect_feature_cols(val_df, user_cols, "USER (val)")
    client_cols = intersect_feature_cols(train_df, client_cols, "CLIENT")
    client_cols = intersect_feature_cols(val_df, client_cols, "CLIENT (val)")

    user_multi_cfg = list(fc.user_multi_cols)
    client_multi_cfg = list(fc.client_multi_cols)
    user_multi = [c for c in user_multi_cfg if c in train_df.columns]
    client_multi = [c for c in client_multi_cfg if c in train_df.columns]
    _missing_um = [c for c in user_multi_cfg if c not in train_df.columns]
    _missing_cm = [c for c in client_multi_cfg if c not in train_df.columns]
    if _missing_um:
        print("USER_MULTI: skipping (not in train):", _missing_um)
    if _missing_cm:
        print("CLIENT_MULTI: skipping (not in train):", _missing_cm)

    user_cat, user_num, _ = split_cat_num_multi(train_df, user_cols, set(user_multi))
    client_cat, client_num, _ = split_cat_num_multi(train_df, client_cols, set(client_multi))

    print("User cats:", len(user_cat), user_cat[:5])
    print("User nums:", len(user_num), user_num[:5])
    print("User multi:", user_multi)
    print("Client cats:", len(client_cat), client_cat)
    print("Client nums:", len(client_num), client_num[:5])
    print("Client multi:", client_multi)

    min_c = tc.min_count
    n_oov = tc.num_oov_buckets
    user_vocabs = {c: build_cat_vocab(train_df[c], min_c, n_oov) for c in user_cat}
    client_vocabs = {c: build_cat_vocab(train_df[c], min_c, n_oov) for c in client_cat}
    user_multi_vocabs = {c: build_multi_token_vocab(train_df[c], min_c, n_oov) for c in user_multi}
    client_multi_vocabs = {c: build_multi_token_vocab(train_df[c], min_c, n_oov) for c in client_multi}

    print("User vocab sizes:", {c: v.size for c, v in user_vocabs.items()})
    print("Client vocab sizes:", {c: v.size for c, v in client_vocabs.items()})
    if user_multi_vocabs:
        print("User multi vocab sizes:", {c: v.size for c, v in user_multi_vocabs.items()})
    if client_multi_vocabs:
        print("Client multi vocab sizes:", {c: v.size for c, v in client_multi_vocabs.items()})

    user_hw: dict[str, np.ndarray] = {}
    client_hw: dict[str, np.ndarray] = {}
    if build_hash_weights:
        pre = int(tc.pretrained_emb_dim)
        if user_vocabs:
            print("Building hash embeddings for user categorical features ...")
            user_hw = build_all_hash_weights(user_vocabs, pre)
        if client_vocabs:
            print("Building hash embeddings for client categorical features ...")
            client_hw = build_all_hash_weights(client_vocabs, pre)
        gc.collect()
        _tu = sum(v.n_frequent for v in user_vocabs.values())
        _tc = sum(v.n_frequent for v in client_vocabs.values())
        print(
            f"Hash tables: user {len(user_hw)} cols / {_tu} frequent tokens; "
            f"client {len(client_hw)} cols / {_tc} frequent tokens; dim={pre}."
        )

    return FeatureArtifacts(
        user_feature_cols=user_cols,
        client_feature_cols=client_cols,
        user_multi_cols=user_multi,
        client_multi_cols=client_multi,
        user_cat_cols=user_cat,
        user_num_cols=user_num,
        client_cat_cols=client_cat,
        client_num_cols=client_num,
        user_vocabs=user_vocabs,
        client_vocabs=client_vocabs,
        user_multi_vocabs=user_multi_vocabs,
        client_multi_vocabs=client_multi_vocabs,
        user_cat_pretrained_weights=user_hw,
        client_cat_pretrained_weights=client_hw,
    )


### Config loader
Parses `train.yaml` / `infer.yaml` into strongly-typed dataclasses.


In [ ]:
# ── Config loader — load_pipeline_config / load_infer_job_config from YAML file ──────────────────────────────────────────────────────
def _parse_inference_section(i: dict, *, path_label: str) -> InferenceConfig:
    for k in (
        "topk_clients",
        "infer_stream_batch_rows",
        "num_physical_gpus",
        "workers_per_gpu",
        "ranking_output",
    ):
        if k not in i or i[k] is None:
            raise KeyError(f"infer.{k} is required in {path_label}")
    return InferenceConfig(
        topk_clients=int(i["topk_clients"]),
        infer_stream_batch_rows=int(i["infer_stream_batch_rows"]),
        num_physical_gpus=int(i["num_physical_gpus"]),
        workers_per_gpu=int(i["workers_per_gpu"]),
        ranking_output=str(i["ranking_output"]),
        rank_user_batch=int(i.get("rank_user_batch", 4096)),
        client_chunk=int(i.get("client_chunk", 8192)),
        use_amp=bool(i.get("use_amp", True)),
        amp_dtype=str(i.get("amp_dtype", "float16")),
        output_min_rows_per_part=int(i.get("output_min_rows_per_part", 100_000)),
        output_parquet_compression=str(i.get("output_parquet_compression", "zstd")),
        debug_cuda=bool(i.get("debug_cuda", False)),
        max_files=int(i["max_files"]) if i.get("max_files") not in (None, "", 0) else None,
        max_users_per_file=int(i["max_users_per_file"])
        if i.get("max_users_per_file") not in (None, "", 0)
        else None,
    )


def load_infer_job_config(path: str | Path) -> InferJobConfig:
    """Load ``configs/infer.yaml`` (standalone inference)."""
    path = Path(path)
    raw: dict[str, Any] = yaml.safe_load(path.read_text(encoding="utf-8")) or {}
    if not isinstance(raw, dict):
        raise ValueError(f"Config root must be a mapping, got {type(raw)}")
    p = raw.get("paths") or {}
    for k in ("infer", "artifacts_base"):
        if k not in p or p[k] is None:
            raise KeyError(f"paths.{k} is required in {path}")
    paths = InferPaths(infer=str(p["infer"]), artifacts_base=str(p["artifacts_base"]))
    infer = _parse_inference_section(raw.get("infer") or {}, path_label=str(path))
    return InferJobConfig(paths=paths, infer=infer)


def load_pipeline_config(path: str | Path) -> PipelineConfig:
    """Load full training pipeline config from YAML (see ``configs/train.yaml``)."""
    path = Path(path)
    raw: dict[str, Any] = yaml.safe_load(path.read_text(encoding="utf-8")) or {}
    if not isinstance(raw, dict):
        raise ValueError(f"Config root must be a mapping, got {type(raw)}")

    p = raw.get("paths") or {}
    for k in ("train", "val", "infer", "artifacts_base"):
        if k not in p or p[k] is None:
            raise KeyError(f"paths.{k} is required in {path}")
    paths = DataPaths(
        train=str(p["train"]),
        val=str(p["val"]),
        infer=str(p["infer"]),
        artifacts_base=str(p["artifacts_base"]),
    )

    f = raw.get("features") or {}
    for k in ("label_col", "device_id_col", "client_id_col"):
        if k not in f or f[k] is None:
            raise KeyError(f"features.{k} is required in {path}")
    features = FeatureConfig(
        label_col=str(f["label_col"]),
        device_id_col=str(f["device_id_col"]),
        client_id_col=str(f["client_id_col"]),
        user_feature_cols=list(f.get("user_feature_cols") or []),
        client_feature_cols=list(f.get("client_feature_cols") or []),
        user_multi_cols=list(f.get("user_multi_cols") or []),
        client_multi_cols=list(f.get("client_multi_cols") or []),
    )

    t = raw.get("train") or {}
    mlp = t.get("mlp_hidden_dims")
    if mlp is None:
        raise KeyError(f"train.mlp_hidden_dims is required in {path}")
    cmh = t.get("client_mlp_hidden")
    if cmh is None:
        cmh = [256, 256]
    train = TrainConfig(
        experiment_name=str(t.get("experiment_name", "two_tower")),
        run_name=t.get("run_name"),
        seed=int(t.get("seed", 42)),
        batch_size=int(t.get("batch_size", 4096)),
        epochs=int(t.get("epochs", 1)),
        lr=float(t.get("lr", 1e-3)),
        weight_decay=float(t.get("weight_decay", 0.0)),
        embed_dim=int(t.get("embed_dim", 1024)),
        dcn_cross_layers=int(t.get("dcn_cross_layers", 3)),
        mlp_hidden_dims=list(mlp),
        min_count=int(t.get("min_count", 5)),
        num_oov_buckets=int(t.get("num_oov_buckets", 1000)),
        multi_max_tokens=int(t.get("multi_max_tokens", 32)),
        num_workers=int(t.get("num_workers", 4)),
        device=str(t.get("device", "cuda")),
        pretrained_emb_dim=int(t.get("pretrained_emb_dim", 128)),
        pretrained_cat_emb_dim=int(t.get("pretrained_cat_emb_dim", 64)),
        freeze_pretrained_base=bool(t.get("freeze_pretrained_base", True)),
        multi_cat_pool=str(t.get("multi_cat_pool", "mean")),
        client_mlp_hidden=list(cmh),
        train_eval_max_examples=int(t.get("train_eval_max_examples", 0) or 0),
    )

    infer = _parse_inference_section(raw.get("infer") or {}, path_label=str(path))

    d = raw.get("data") or {}
    row_filter = d.get("single_client_row_filter")
    if row_filter is not None and not isinstance(row_filter, dict):
        raise TypeError("data.single_client_row_filter must be a mapping or null")
    hardcoded = d.get("single_client_features_hardcoded")
    if hardcoded is not None and not isinstance(hardcoded, dict):
        raise TypeError("data.single_client_features_hardcoded must be a mapping or null")

    data_load = DataLoadConfig(
        merge_client_metadata=bool(d.get("merge_client_metadata", False)),
        client_metadata_uri=d.get("client_metadata_uri"),
        inject_single_client_metadata=bool(d.get("inject_single_client_metadata", False)),
        single_client_metadata_uri=d.get("single_client_metadata_uri"),
        single_client_row_filter=dict(row_filter) if row_filter else None,
        single_client_features_hardcoded=dict(hardcoded) if hardcoded else None,
    )

    extra = raw.get("extra")
    if extra is not None and not isinstance(extra, dict):
        raise TypeError("extra must be a mapping or null")

    return PipelineConfig(
        paths=paths,
        features=features,
        train=train,
        infer=infer,
        data_load=data_load,
        mlflow_tracking_uri=raw.get("mlflow_tracking_uri"),
        extra=dict(extra) if extra else None,
    )


### Notebook YAML helper
Converts the YAML string above into a `PipelineConfig` object (writes to a temp file and calls `load_pipeline_config`).


In [ ]:
# ── YAML string loader (notebook helper) ─────────────────────────────────
def load_pipeline_config_from_yaml_string(s: str) -> PipelineConfig:
    """Write YAML to a temp file and parse with load_pipeline_config()."""
    import tempfile
    d = Path(tempfile.mkdtemp())
    p = d / "train.yaml"
    p.write_text(s, encoding="utf-8")
    return load_pipeline_config(p)


### Data loading
Reads train / validation Parquet from S3 or local disk. Optionally joins client metadata by `client_id_col`.


In [ ]:
# ── Data loading — load_train_validation_frames, merge_client_metadata_into_frames ──────────────────────────────────────────────────────
def merge_client_metadata_into_frames(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    cfg: PipelineConfig,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Left-join ``client_*`` columns from client-metadata Parquet onto train/val by ``client_id_col``.

    Expects the same layout as the SQL ``client_metadata`` table (key column ``client_bundle_id``
    unless ``features.client_id_col`` is set to another name present in both frames and metadata).
    """
    dl = cfg.data_load
    if not dl.client_metadata_uri:
        raise ValueError("merge_client_metadata: set data.client_metadata_uri in config")

    key = cfg.features.client_id_col
    if key not in train_df.columns:
        raise KeyError(f"train data missing join column {key!r}")
    if key not in val_df.columns:
        raise KeyError(f"val data missing join column {key!r}")

    cmf = pd.read_parquet(dl.client_metadata_uri)
    if key not in cmf.columns:
        raise KeyError(f"client_metadata missing join column {key!r}; have: {list(cmf.columns)[:40]}")

    need = list(cfg.features.client_feature_cols)
    missing_meta = [c for c in need if c not in cmf.columns]
    if missing_meta:
        raise KeyError(
            "client_metadata missing configured client_feature_cols: "
            f"{missing_meta[:20]}{'...' if len(missing_meta) > 20 else ''}"
        )

    meta = cmf[[key, *need]].drop_duplicates(subset=[key], keep="first")

    def _apply(df: pd.DataFrame, name: str) -> pd.DataFrame:
        out = df.drop(columns=[c for c in need if c in df.columns], errors="ignore")
        merged = out.merge(meta, on=key, how="left", validate="many_to_one")
        bad = merged[need].isna().any(axis=1)
        if bad.any():
            missing_ids = merged.loc[bad, key].drop_duplicates().tolist()
            sample = missing_ids[:15]
            raise ValueError(
                f"{name}: {int(bad.sum())} rows have no client_metadata match for {key!r}; "
                f"example ids: {sample}{'...' if len(missing_ids) > 15 else ''}"
            )
        return merged

    train_m = _apply(train_df, "train")
    val_m = _apply(val_df, "val")
    print(
        f"[merge_client] joined {len(need)} client columns from {dl.client_metadata_uri!r} "
        f"on {key!r}; train={len(train_m):,} val={len(val_m):,}"
    )
    return train_m, val_m


def _resolve_injected_client_id(crow: pd.Series, cfg: PipelineConfig) -> tuple[str | None, object | None]:
    candidates = [
        cfg.features.client_id_col,
        "client_id",
        "client_bundle_id",
        
    ]
    for col in candidates:
        if col and col in crow.index:
            return col, crow[col]

    row_filter = cfg.data_load.single_client_row_filter or {}
    for col in candidates:
        if col and col in row_filter:
            return cfg.features.client_id_col, row_filter[col]

    return None, None


def load_train_validation_frames(cfg: PipelineConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Load train and validation Parquet from S3 or local paths (``pd.read_parquet``).

    Optional:

    - **Per-row** client features via ``merge_client_metadata`` + ``client_metadata_uri``
      (join on ``features.client_id_col``).
    - **Single-client** broadcast when ``inject_single_client_metadata`` is True.
    """
    train_df = pd.read_parquet(cfg.paths.train)
    val_df = pd.read_parquet(cfg.paths.val)

    dl = cfg.data_load
    if dl.merge_client_metadata and dl.inject_single_client_metadata:
        raise ValueError(
            "Use either data.merge_client_metadata or data.inject_single_client_metadata, not both."
        )
    if dl.merge_client_metadata:
        train_df, val_df = merge_client_metadata_into_frames(train_df, val_df, cfg)

    if dl.inject_single_client_metadata:
        if dl.single_client_metadata_uri:
            cmf = pd.read_parquet(dl.single_client_metadata_uri)
            if dl.single_client_row_filter:
                for fk, fv in dl.single_client_row_filter.items():
                    if fk not in cmf.columns:
                        raise KeyError(
                            f"metadata missing filter column {fk!r}; "
                            f"have: {list(cmf.columns)[:30]} ..."
                        )
                    cmf = cmf[cmf[fk] == fv]
            if len(cmf) == 0:
                raise ValueError("inject_single_client_metadata: no row left after single_client_row_filter")
            crow = cmf.iloc[0]
        elif dl.single_client_features_hardcoded:
            crow = pd.Series(dl.single_client_features_hardcoded)
        else:
            raise ValueError(
                "inject_single_client_metadata is True: set single_client_metadata_uri or "
                "single_client_features_hardcoded in config data section"
            )
        inj = 0
        for col in cfg.features.client_feature_cols:
            if col not in crow.index:
                continue
            train_df[col] = crow[col]
            val_df[col] = crow[col]
            inj += 1

        injected_id_col, injected_id_val = _resolve_injected_client_id(crow, cfg)
        if injected_id_col is not None:
            train_df[cfg.features.client_id_col] = injected_id_val
            val_df[cfg.features.client_id_col] = injected_id_val
            print(
                f"[inject_client] set client id column {cfg.features.client_id_col!r} "
                f"from metadata field {injected_id_col!r}"
            )
        print(
            f"[inject_client] broadcast {inj} client columns from one metadata row "
            f"onto train={len(train_df):,} val={len(val_df):,} rows"
        )

    if cfg.features.label_col not in train_df.columns:
        raise KeyError(
            f"Train data missing label column {cfg.features.label_col!r}. "
            f"Columns: {list(train_df.columns)[:40]} ..."
        )

    print("Train shape:", train_df.shape)
    print("Val shape:", val_df.shape)
    return train_df, val_df


### Dataset
`TwoTowerDataset` pre-encodes the entire DataFrame into tensors once, then serves `(user_cat, user_num, …, label)` tuples.


In [ ]:
# ── TwoTowerDataset — pre-encodes train / val frames into tensors ──────────────────────────────────────────────────────
from torch.utils.data import Dataset



class TwoTowerDataset(Dataset):
    """Pre-encodes the full frame once (same as reference notebook)."""

    def __init__(self, df: pd.DataFrame, fa: FeatureArtifacts, *, label_col: str, multi_max_tokens: int):
        df = df.reset_index(drop=True)
        self.user_cat = encode_cats(df, fa.user_cat_cols, fa.user_vocabs)
        self.user_num = encode_nums(df, fa.user_num_cols)
        self.user_multi = encode_multi_matrix(df, fa.user_multi_cols, fa.user_multi_vocabs, multi_max_tokens)
        self.client_cat = encode_cats(df, fa.client_cat_cols, fa.client_vocabs)
        self.client_num = encode_nums(df, fa.client_num_cols)
        self.client_multi = encode_multi_matrix(
            df, fa.client_multi_cols, fa.client_multi_vocabs, multi_max_tokens
        )
        self.y = torch.from_numpy(df[label_col].astype("float32").to_numpy()).unsqueeze(1)

    def __len__(self) -> int:
        return int(self.y.shape[0])

    def __getitem__(self, idx: int):
        return (
            self.user_cat[idx],
            self.user_num[idx],
            self.user_multi[idx],
            self.client_cat[idx],
            self.client_num[idx],
            self.client_multi[idx],
            self.y[idx],
        )


### Model architecture
- **`DCNv2UserTower`**: DCN-v2 cross network + MLP deep stack for user features.
- **`ClientMLPTower`**: MLP tower for client (advertiser app) features.
- **`TwoTowerModel`**: combines both towers; dot-product logit with learned temperature.
- **`build_two_tower_model(fa, cfg)`**: factory that wires vocabs → embedding sizes → model.


In [ ]:
# ── Model architecture — CatEmbedder, DCNv2UserTower, ClientMLPTower, TwoTowerModel, build_two_tower_model ──────────────────────────────────────────────────────
def embedding_dim_for_cardinality(vocab_size: int) -> int:
    v = int(max(1, vocab_size))
    if v < 100:
        return 6
    if v < 10_000:
        return 20
    if v < 1_000_000:
        return 64
    return 128


class CatEmbedder(nn.Module):
    """Categorical embedder: random init or hash-pretrained + linear projection."""

    def __init__(
        self,
        vocab_sizes: List[int],
        use_pretrained: bool = False,
        pretrained_weights: list | None = None,
        pretrained_emb_dim: int = 384,
        target_emb_dim_per_col: int = 64,
        freeze_base: bool = False,
    ):
        super().__init__()
        self.use_pretrained = use_pretrained

        if use_pretrained:
            self.emb_dims = [target_emb_dim_per_col] * len(vocab_sizes)
            self.embs = nn.ModuleList([nn.Embedding(int(v), pretrained_emb_dim, padding_idx=0) for v in vocab_sizes])
            if pretrained_weights is not None:
                for emb, w_np in zip(self.embs, pretrained_weights):
                    w = torch.from_numpy(w_np).float()
                    n = min(w.shape[0], emb.weight.shape[0])
                    emb.weight.data[:n].copy_(w[:n])
            if freeze_base:
                for emb in self.embs:
                    emb.weight.requires_grad_(False)
            self.projs: nn.ModuleList | None = nn.ModuleList(
                [nn.Linear(pretrained_emb_dim, target_emb_dim_per_col, bias=False) for _ in vocab_sizes]
            )
        else:
            self.emb_dims = [embedding_dim_for_cardinality(v) for v in vocab_sizes]
            self.embs = nn.ModuleList([nn.Embedding(int(v), int(d)) for v, d in zip(vocab_sizes, self.emb_dims)])
            self.projs = None

    def output_dim(self) -> int:
        return int(sum(self.emb_dims))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.numel() == 0:
            return torch.zeros((x.shape[0], 0), device=x.device)
        if self.use_pretrained and self.projs is not None:
            parts = [proj(emb(x[:, i])) for i, (emb, proj) in enumerate(zip(self.embs, self.projs))]
        else:
            parts = [emb(x[:, i]) for i, emb in enumerate(self.embs)]
        return torch.cat(parts, dim=1)


class TokenSeqAttentionPool(nn.Module):
    def __init__(self, d: int):
        super().__init__()
        self.score = nn.Linear(int(d), 1)

    def forward(self, e: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        s = self.score(e).squeeze(-1)
        s = s.masked_fill(mask <= 0, -1e9)
        w = torch.softmax(s, dim=1).unsqueeze(-1)
        return (e * w).sum(dim=1)


class PooledMultiCatEmbedder(nn.Module):
    def __init__(self, vocab_sizes: List[int], emb_dims: List[int], pool: str = "mean"):
        super().__init__()
        self.pool = pool
        self.emb_dims = [int(d) for d in emb_dims]
        self.embs = nn.ModuleList(
            [nn.Embedding(int(v), int(d), padding_idx=0) for v, d in zip(vocab_sizes, self.emb_dims)]
        )
        self._attn: nn.ModuleList | None = None
        if pool == "attention":
            self._attn = nn.ModuleList([TokenSeqAttentionPool(d) for d in self.emb_dims])

    def output_dim(self) -> int:
        return int(sum(self.emb_dims))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.shape[1] == 0:
            return torch.zeros((x.shape[0], 0), device=x.device)
        outs = []
        for i, emb in enumerate(self.embs):
            ids = x[:, i, :]
            e = emb(ids)
            mask = (ids != 0).float()
            if self.pool == "sum":
                pooled = (e * mask.unsqueeze(-1)).sum(dim=1)
            elif self.pool == "attention" and self._attn is not None:
                pooled = self._attn[i](e, mask)
            else:
                denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
                pooled = (e * mask.unsqueeze(-1)).sum(dim=1) / denom
            outs.append(pooled)
        return torch.cat(outs, dim=1)


def _safe_l2_normalize(x: torch.Tensor, dim: int = 1, eps: float = 1e-6) -> torch.Tensor:
    denom = x.norm(p=2, dim=dim, keepdim=True).clamp_min(eps)
    return x / denom


class DCNv2CrossLayer(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(input_dim, input_dim) * 0.01)
        self.bias = nn.Parameter(torch.zeros(input_dim))

    def forward(self, x0: torch.Tensor, xl: torch.Tensor) -> torch.Tensor:
        cross = xl @ self.weight
        cross = cross + self.bias
        cross = x0 * cross
        return cross + xl


class DCNv2UserTower(nn.Module):
    def __init__(
        self,
        user_vocab_sizes: List[int],
        user_num_dim: int,
        emb_dim: int = 1024,
        num_cross_layers: int = 3,
        deep_hidden: List[int] | None = None,
        user_multi_vocab_sizes: List[int] | None = None,
        user_multi_emb_dims: List[int] | None = None,
        multi_pool: str = "mean",
        use_pretrained_cat: bool = False,
        user_cat_pretrained_weights: list | None = None,
        pretrained_emb_dim: int = 384,
        target_cat_emb_dim: int = 64,
        freeze_base: bool = False,
    ):
        super().__init__()
        if deep_hidden is None:
            deep_hidden = [512, 512]

        self.user_cat = CatEmbedder(
            user_vocab_sizes,
            use_pretrained=use_pretrained_cat,
            pretrained_weights=user_cat_pretrained_weights,
            pretrained_emb_dim=pretrained_emb_dim,
            target_emb_dim_per_col=target_cat_emb_dim,
            freeze_base=freeze_base,
        )
        ms = list(user_multi_vocab_sizes or [])
        if ms:
            ed = user_multi_emb_dims or [embedding_dim_for_cardinality(v) for v in ms]
            self.user_multi = PooledMultiCatEmbedder(ms, ed, pool=multi_pool)
        else:
            self.user_multi = None

        cat_w = self.user_cat.output_dim() + (self.user_multi.output_dim() if self.user_multi else 0)
        input_dim = int(cat_w + user_num_dim)

        self.cross_layers = nn.ModuleList([DCNv2CrossLayer(input_dim) for _ in range(num_cross_layers)])

        deep_layers: list[nn.Module] = []
        prev = input_dim
        for h in deep_hidden:
            deep_layers.append(nn.Linear(prev, h))
            deep_layers.append(nn.ReLU())
            deep_layers.append(nn.BatchNorm1d(h))
            prev = h
        self.deep = nn.Sequential(*deep_layers)

        self.fc = nn.Linear(prev + input_dim, emb_dim)

    def forward(self, user_cat: torch.Tensor, user_num: torch.Tensor, user_multi: torch.Tensor) -> torch.Tensor:
        cat = self.user_cat(user_cat)
        if self.user_multi is not None:
            cat = torch.cat([cat, self.user_multi(user_multi)], dim=1)
        x0 = torch.cat([cat, user_num], dim=1)
        xl = x0
        for layer in self.cross_layers:
            xl = layer(x0, xl)
        cross_out = xl
        deep_out = self.deep(x0)
        emb = self.fc(torch.cat([cross_out, deep_out], dim=1))
        return _safe_l2_normalize(emb, dim=1)


class ClientMLPTower(nn.Module):
    def __init__(
        self,
        client_vocab_sizes: List[int],
        client_num_dim: int,
        emb_dim: int = 1024,
        hidden: List[int] | None = None,
        client_multi_vocab_sizes: List[int] | None = None,
        client_multi_emb_dims: List[int] | None = None,
        multi_pool: str = "mean",
        use_pretrained_cat: bool = False,
        client_cat_pretrained_weights: list | None = None,
        pretrained_emb_dim: int = 384,
        target_cat_emb_dim: int = 64,
        freeze_base: bool = False,
    ):
        super().__init__()
        if hidden is None:
            hidden = [256, 256]

        self.client_cat = CatEmbedder(
            client_vocab_sizes,
            use_pretrained=use_pretrained_cat,
            pretrained_weights=client_cat_pretrained_weights,
            pretrained_emb_dim=pretrained_emb_dim,
            target_emb_dim_per_col=target_cat_emb_dim,
            freeze_base=freeze_base,
        )
        ms = list(client_multi_vocab_sizes or [])
        if ms:
            ed = client_multi_emb_dims or [embedding_dim_for_cardinality(v) for v in ms]
            self.client_multi = PooledMultiCatEmbedder(ms, ed, pool=multi_pool)
        else:
            self.client_multi = None

        cat_w = self.client_cat.output_dim() + (self.client_multi.output_dim() if self.client_multi else 0)
        input_dim = int(cat_w + client_num_dim)

        layers: list[nn.Module] = []
        prev = input_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.LayerNorm(h))
            prev = h
        layers.append(nn.Linear(prev, emb_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, client_cat: torch.Tensor, client_num: torch.Tensor, client_multi: torch.Tensor) -> torch.Tensor:
        cat = self.client_cat(client_cat)
        if self.client_multi is not None:
            cat = torch.cat([cat, self.client_multi(client_multi)], dim=1)
        x = torch.cat([cat, client_num], dim=1)
        emb = self.net(x)
        return _safe_l2_normalize(emb, dim=1)


class TwoTowerModel(nn.Module):
    def __init__(
        self,
        user_vocab_sizes: List[int],
        user_num_dim: int,
        client_vocab_sizes: List[int],
        client_num_dim: int,
        emb_dim: int = 1024,
        user_multi_vocab_sizes: List[int] | None = None,
        user_multi_emb_dims: List[int] | None = None,
        client_multi_vocab_sizes: List[int] | None = None,
        client_multi_emb_dims: List[int] | None = None,
        multi_pool: str = "mean",
        use_pretrained_cat: bool = False,
        user_cat_pretrained_weights: list | None = None,
        client_cat_pretrained_weights: list | None = None,
        pretrained_emb_dim: int = 384,
        target_cat_emb_dim: int = 64,
        freeze_pretrained_base: bool = False,
        num_cross_layers: int = 2,
        user_deep_hidden: List[int] | None = None,
        client_hidden: List[int] | None = None,
    ):
        super().__init__()
        self.log_scale = nn.Parameter(torch.tensor(math.log(20.0)))
        self.user_tower = DCNv2UserTower(
            user_vocab_sizes,
            user_num_dim,
            emb_dim=emb_dim,
            num_cross_layers=num_cross_layers,
            deep_hidden=user_deep_hidden,
            user_multi_vocab_sizes=user_multi_vocab_sizes,
            user_multi_emb_dims=user_multi_emb_dims,
            multi_pool=multi_pool,
            use_pretrained_cat=use_pretrained_cat,
            user_cat_pretrained_weights=user_cat_pretrained_weights,
            pretrained_emb_dim=pretrained_emb_dim,
            target_cat_emb_dim=target_cat_emb_dim,
            freeze_base=freeze_pretrained_base,
        )
        self.client_tower = ClientMLPTower(
            client_vocab_sizes,
            client_num_dim,
            emb_dim=emb_dim,
            hidden=client_hidden,
            client_multi_vocab_sizes=client_multi_vocab_sizes,
            client_multi_emb_dims=client_multi_emb_dims,
            multi_pool=multi_pool,
            use_pretrained_cat=use_pretrained_cat,
            client_cat_pretrained_weights=client_cat_pretrained_weights,
            pretrained_emb_dim=pretrained_emb_dim,
            target_cat_emb_dim=target_cat_emb_dim,
            freeze_base=freeze_pretrained_base,
        )

    def forward(
        self,
        user_cat: torch.Tensor,
        user_num: torch.Tensor,
        client_cat: torch.Tensor,
        client_num: torch.Tensor,
        user_multi: torch.Tensor,
        client_multi: torch.Tensor,
    ):
        user_emb = self.user_tower(user_cat, user_num, user_multi)
        client_emb = self.client_tower(client_cat, client_num, client_multi)
        scale = self.log_scale.clamp(math.log(1.0), math.log(100.0)).exp()
        logits = (user_emb * client_emb).sum(dim=1, keepdim=True) * scale
        return logits, user_emb, client_emb


def build_two_tower_model(fa, cfg) -> TwoTowerModel:
    """Construct model from feature artifacts + pipeline config."""
    tc = cfg.train
    user_vocab_sizes = [fa.user_vocabs[c].size for c in fa.user_cat_cols]
    client_vocab_sizes = [fa.client_vocabs[c].size for c in fa.client_cat_cols]
    user_multi_vocab_sizes = [fa.user_multi_vocabs[c].size for c in fa.user_multi_cols]
    client_multi_vocab_sizes = [fa.client_multi_vocabs[c].size for c in fa.client_multi_cols]
    user_multi_emb_dims = [embedding_dim_for_cardinality(v) for v in user_multi_vocab_sizes]
    client_multi_emb_dims = [embedding_dim_for_cardinality(v) for v in client_multi_vocab_sizes]

    user_num_dim = len(fa.user_num_cols)
    client_num_dim = len(fa.client_num_cols)

    use_pt = bool(fa.user_cat_pretrained_weights) and bool(fa.client_cat_pretrained_weights)
    user_w = [fa.user_cat_pretrained_weights[c] for c in fa.user_cat_cols] if use_pt and fa.user_cat_cols else None
    client_w = (
        [fa.client_cat_pretrained_weights[c] for c in fa.client_cat_cols] if use_pt and fa.client_cat_cols else None
    )

    return TwoTowerModel(
        user_vocab_sizes=user_vocab_sizes,
        user_num_dim=user_num_dim,
        client_vocab_sizes=client_vocab_sizes,
        client_num_dim=client_num_dim,
        emb_dim=tc.embed_dim,
        num_cross_layers=tc.dcn_cross_layers,
        user_deep_hidden=list(tc.mlp_hidden_dims),
        client_hidden=list(tc.client_mlp_hidden),
        user_multi_vocab_sizes=user_multi_vocab_sizes or None,
        user_multi_emb_dims=user_multi_emb_dims or None,
        client_multi_vocab_sizes=client_multi_vocab_sizes or None,
        client_multi_emb_dims=client_multi_emb_dims or None,
        multi_pool=tc.multi_cat_pool,
        use_pretrained_cat=use_pt,
        user_cat_pretrained_weights=user_w,
        client_cat_pretrained_weights=client_w,
        pretrained_emb_dim=tc.pretrained_emb_dim,
        target_cat_emb_dim=tc.pretrained_cat_emb_dim,
        freeze_pretrained_base=tc.freeze_pretrained_base,
    )


### Training helpers
Small utility functions used inside `train_and_log`: device resolution, binary metrics (AUC-ROC, AUC-PR, F1, …), artifact writing.


In [ ]:
# ── Training helpers ─────────────────────────────────────────────────────

def _resolve_device(train_device: str) -> torch.device:
    if train_device == "cuda" and not torch.cuda.is_available():
        return torch.device("cpu")
    return torch.device(train_device)


def _write_bytes(uri: str, data: bytes) -> None:
    """Write bytes to S3 or a local file."""
    if uri.startswith("s3://"):
        import s3fs
        with s3fs.S3FileSystem().open(uri, "wb") as f:
            f.write(data)
    else:
        p = Path(uri)
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_bytes(data)


def _resolve_client_id_column(train_df: pd.DataFrame, preferred: str) -> str:
    candidates = [preferred, "client_id", "client_bundle_id",
                  "client_bundle", "clientId", "bundle_id", "client"]
    for c in candidates:
        if c in train_df.columns:
            return c
    raise KeyError(
        "Could not find client id column in training data. "
        f"Tried: {candidates}. Available: {list(train_df.columns)[:50]} ..."
    )


def _sigmoid_prob_from_logits(y_score: np.ndarray) -> np.ndarray:
    z = np.clip(np.asarray(y_score, dtype=np.float64), -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-z))


def _binary_metrics(
    y_true: np.ndarray,
    y_score: np.ndarray,
    *,
    threshold: float = 0.5,
) -> dict[str, float]:
    """AUC-ROC, AUC-PR, precision, recall, F1, accuracy, log-loss, confusion matrix."""
    from sklearn.metrics import (
        accuracy_score, average_precision_score, confusion_matrix,
        f1_score, log_loss, precision_score, recall_score, roc_auc_score,
    )
    yt = np.asarray(y_true).astype(np.int32).ravel()
    ys = np.asarray(y_score, dtype=np.float64).ravel()
    prob = _sigmoid_prob_from_logits(ys)
    pred = (prob >= float(threshold)).astype(np.int32)
    out: dict[str, float] = {}
    try:
        out["auc_roc"] = float(roc_auc_score(yt, ys))
    except ValueError:
        out["auc_roc"] = float("nan")
    try:
        out["auc_pr"] = float(average_precision_score(yt, ys))
    except ValueError:
        out["auc_pr"] = float("nan")
    out["precision"] = float(precision_score(yt, pred, zero_division=0))
    out["recall"] = float(recall_score(yt, pred, zero_division=0))
    out["f1"] = float(f1_score(yt, pred, zero_division=0))
    out["accuracy"] = float(accuracy_score(yt, pred))
    try:
        out["logloss"] = float(log_loss(yt, prob, labels=[0, 1]))
    except ValueError:
        out["logloss"] = float("nan")
    out["label_pos_rate"] = float(yt.mean()) if yt.size else float("nan")
    cm = confusion_matrix(yt, pred, labels=[0, 1])
    out["tn"] = float(cm[0, 0])
    out["fp"] = float(cm[0, 1])
    out["fn"] = float(cm[1, 0])
    out["tp"] = float(cm[1, 1])
    return out


### `train_and_log()`
The main training loop:
1. Builds `DataLoader`s from `TwoTowerDataset`.
2. Constructs the `TwoTowerModel`.
3. Runs `epochs` of BCE-logits training; evaluates on validation each epoch.
4. Saves the **best-AUC checkpoint** (user tower `.pt`, vocab `.pkl`, client embeddings `.parquet`) to `paths.artifacts_base`.
5. Logs params, per-epoch metrics, and the full model to MLflow.


In [ ]:
# ── train_and_log() — full training loop with MLflow logging ──────────────────────────────────────────────────────
def train_and_log(
    *,
    cfg: Union[PipelineConfig, str, Path],
    train_df: pd.DataFrame | None = None,
    val_df: pd.DataFrame | None = None,
    feature_artifacts: FeatureArtifacts | None = None,
) -> None:
    runlog = start_run_log(kind="train", name="two_tower"    t_start = time.time(    if not isinstance(cfg, PipelineConfig):
        cfg = load_pipeline_config(cfg
    if train_df is None or val_df is None:
        train_df, val_df = load_train_validation_frames(cfg
    print(f"[train_and_log] train={train_df.shape} val={val_df.shape}"    print(f"[train_and_log] run log: {runlog.path}"    runlog.write(f"CONFIG experiment={cfg.train.experiment_name} device={cfg.train.device} epochs={cfg.train.epochs}"    runlog.write(f"DATA train_rows={train_df.shape[0]} val_rows={val_df.shape[0]}"
    if feature_artifacts is None:
        feature_artifacts = prepare_training_features(train_df, val_df, cfg
    fa = feature_artifacts
    tc = cfg.train
    fc = cfg.features

    torch.manual_seed(tc.seed    np.random.seed(tc.seed % (2**32 - 1)
    device = _resolve_device(tc.device    train_ds = TwoTowerDataset(train_df, fa, label_col=fc.label_col, multi_max_tokens=tc.multi_max_tokens    val_ds = TwoTowerDataset(val_df, fa, label_col=fc.label_col, multi_max_tokens=tc.multi_max_tokens
    train_loader = DataLoader(
        train_ds,
        batch_size=tc.batch_size,
        shuffle=True,
        num_workers=tc.num_workers,
        collate_fn=collate_fn,
        val_loader = DataLoader(
        val_ds,
        batch_size=tc.batch_size,
        shuffle=False,
        num_workers=tc.num_workers,
        collate_fn=collate_fn,
    
    user_vocab_sizes = [fa.user_vocabs[c].size for c in fa.user_cat_cols]
    client_vocab_sizes = [fa.client_vocabs[c].size for c in fa.client_cat_cols]
    user_multi_vocab_sizes = [fa.user_multi_vocabs[c].size for c in fa.user_multi_cols]
    client_multi_vocab_sizes = [fa.client_multi_vocabs[c].size for c in fa.client_multi_cols]
    user_multi_emb_dims = [embedding_dim_for_cardinality(v) for v in user_multi_vocab_sizes]
    client_multi_emb_dims = [embedding_dim_for_cardinality(v) for v in client_multi_vocab_sizes]
    user_num_dim = len(fa.user_num_cols    client_num_dim = len(fa.client_num_cols    user_cat_out = sum(embedding_dim_for_cardinality(v) for v in user_vocab_sizes    client_cat_out = sum(embedding_dim_for_cardinality(v) for v in client_vocab_sizes    user_input_dim = user_cat_out + sum(user_multi_emb_dims) + user_num_dim
    client_input_dim = client_cat_out + sum(client_multi_emb_dims) + client_num_dim

    use_pt = bool(fa.user_cat_pretrained_weights) and bool(fa.client_cat_pretrained_weights    model = build_two_tower_model(fa, cfg).to(device
    if use_pt:
        n_frozen = sum(not any(p.requires_grad for p in emb.parameters()) for emb in model.user_tower.user_cat.embs        print(
            f"Model: pretrained cat embeddings ({'frozen' if tc.freeze_pretrained_base else 'fine-tunable'} base, "
            f"{tc.pretrained_cat_emb_dim}-dim projection). Frozen user emb tables: {n_frozen}/{len(model.user_tower.user_cat.embs)}."
            else:
        print("Model: random-init categorical embeddings."
    criterion = nn.BCEWithLogitsLoss(    optimizer = torch.optim.Adam(model.parameters(), lr=tc.lr, weight_decay=tc.weight_decay
    setup_mlflow(cfg.mlflow_tracking_uri, tc.experiment_name
    try:
        with mlflow.start_run(run_name=tc.run_name):
            mlflow.log_params(
                {
                    "batch_size": tc.batch_size,
                    "lr": tc.lr,
                    "weight_decay": tc.weight_decay,
                    "epochs": tc.epochs,
                    "emb_dim": tc.embed_dim,
                    "num_cross_layers": tc.dcn_cross_layers,
                    "user_deep_hidden": str(list(tc.mlp_hidden_dims)),
                    "client_mlp_hidden": str(list(tc.client_mlp_hidden)),
                    "log_scale_init": round(math.log(20.0), 4),
                    "user_input_dim": user_input_dim,
                    "client_input_dim": client_input_dim,
                    "min_token_count": tc.min_count,
                    "num_oov_buckets": tc.num_oov_buckets,
                    "multi_cat_max_tokens": tc.multi_max_tokens,
                    "multi_cat_pool": tc.multi_cat_pool,
                    "pretrained_emb_dim": tc.pretrained_emb_dim,
                    "pretrained_cat_emb_dim": tc.pretrained_cat_emb_dim,
                    "freeze_pretrained_base": tc.freeze_pretrained_base,
                    "use_pretrained_cat": use_pt,
                }
            
            best_val_auc = float("-inf"            best_epoch = -1
            best_snapshot: dict | None = None

            last_hb = time.time(            hb_every_s = 300.0  # 5 minutes
            for epoch in range(tc.epochs):
                model.train(                total_loss = 0.0
                total_count = 0
                # Optional: compute train AUC/precision/recall on a bounded sample of seen examples.
                # This helps diagnose overfitting (train >> val) without storing the full epoch.
                train_eval_max = int(getattr(tc, "train_eval_max_examples", 0) or 0                train_eval_max = max(train_eval_max, 0                train_scores: list[np.ndarray] = []
                train_labels: list[np.ndarray] = []
                train_eval_kept = 0

                for batch in train_loader:
                    user_cat = batch.user_cat.to(device                    user_num = batch.user_num.to(device                    user_multi = batch.user_multi.to(device                    client_cat = batch.client_cat.to(device                    client_num = batch.client_num.to(device                    client_multi = batch.client_multi.to(device                    labels = batch.label.to(device
                    optimizer.zero_grad(                    logits, _, _ = model(user_cat, user_num, client_cat, client_num, user_multi, client_multi                    loss = criterion(logits, labels                    loss.backward(                    optimizer.step(
                    bs = labels.size(0                    total_loss += loss.item() * bs
                    total_count += bs

                    if train_eval_max > 0 and train_eval_kept < train_eval_max:
                        # Keep up to train_eval_max examples (first-N) for stable, cheap epoch metrics.
                        keep = min(bs, train_eval_max - train_eval_kept                        if keep > 0:
                            train_scores.append(logits.detach().cpu().numpy().ravel()[:keep]                            train_labels.append(labels.detach().cpu().numpy().ravel()[:keep]                            train_eval_kept += keep
                    now = time.time(                    if now - last_hb >= hb_every_s:
                        runlog.write(
                            f"HEARTBEAT epoch={epoch + 1}/{tc.epochs} seen={total_count} "
                            f"avg_loss={total_loss / max(total_count, 1):.6f}"
                                                last_hb = now

                avg_train_loss = total_loss / max(total_count, 1                train_metrics: dict[str, float] | None = None
                if train_eval_max > 0 and train_eval_kept > 0:
                    ytr = np.concatenate(train_labels, axis=0                    yts = np.concatenate(train_scores, axis=0                    train_metrics = _binary_metrics(ytr, yts, threshold=0.5
                model.eval(                val_loss = 0.0
                val_count = 0
                val_uemb_bad = 0
                val_cemb_bad = 0
                val_logits_nonfinite = 0
                all_logits: list[torch.Tensor] = []
                all_labels: list[torch.Tensor] = []
                with torch.no_grad():
                    for batch in val_loader:
                        user_cat = batch.user_cat.to(device                        user_num = batch.user_num.to(device                        user_multi = batch.user_multi.to(device                        client_cat = batch.client_cat.to(device                        client_num = batch.client_num.to(device                        client_multi = batch.client_multi.to(device                        labels = batch.label.to(device
                        logits, uemb, cemb = model(
                            user_cat, user_num, client_cat, client_num, user_multi, client_multi
                                                loss = criterion(logits, labels
                        bs = labels.size(0                        val_uemb_bad += int((~torch.isfinite(uemb)).any(dim=1).sum().item()                        val_cemb_bad += int((~torch.isfinite(cemb)).any(dim=1).sum().item()                        val_logits_nonfinite += int((~torch.isfinite(logits)).sum().item()                        val_loss += loss.item() * bs
                        val_count += bs
                        all_logits.append(logits.cpu()                        all_labels.append(labels.cpu()
                avg_val_loss = val_loss / max(val_count, 1                y_true = torch.cat(all_labels).numpy().ravel(                y_score = torch.cat(all_logits).numpy().ravel(                val_metrics = _binary_metrics(y_true, y_score, threshold=0.5                val_auc = float(val_metrics["auc_roc"]
                if np.isfinite(val_auc) and val_auc > best_val_auc:
                    best_val_auc = val_auc
                best_epoch = epoch
                best_snapshot = {
                    "user_tower": {k: v.detach().cpu().clone() for k, v in model.user_tower.state_dict().items()},
                    "client_tower": {k: v.detach().cpu().clone() for k, v in model.client_tower.state_dict().items()},
                    "log_scale": model.log_scale.detach().cpu().clone(),
                }
                print(f"  -> new best val_auc={best_val_auc:.4f} (epoch {epoch + 1})"
            _scale_val = model.log_scale.clamp(math.log(1.0), math.log(100.0)).exp().item(            metrics_to_log = {
                "train_loss": avg_train_loss,
                "val_loss": avg_val_loss,
                "val_uemb_nonfinite_rows": float(val_uemb_bad),
                "val_cemb_nonfinite_rows": float(val_cemb_bad),
                "val_logits_nonfinite_cells": float(val_logits_nonfinite),
                "temperature_scale": _scale_val,
                # validation (rich                "val_auc": float(val_metrics["auc_roc"]),
                "val_auc_pr": float(val_metrics["auc_pr"]),
                "val_precision": float(val_metrics["precision"]),
                "val_recall": float(val_metrics["recall"]),
                "val_f1": float(val_metrics["f1"]),
                "val_accuracy": float(val_metrics["accuracy"]),
                "val_logloss": float(val_metrics["logloss"]),
                "val_label_pos_rate": float(val_metrics["label_pos_rate"]),
                "val_tn": float(val_metrics["tn"]),
                "val_fp": float(val_metrics["fp"]),
                "val_fn": float(val_metrics["fn"]),
                "val_tp": float(val_metrics["tp"]),
            }
            if train_metrics is not None:
                metrics_to_log.update(
                    {
                        "train_auc": float(train_metrics["auc_roc"]),
                        "train_auc_pr": float(train_metrics["auc_pr"]),
                        "train_precision": float(train_metrics["precision"]),
                        "train_recall": float(train_metrics["recall"]),
                        "train_f1": float(train_metrics["f1"]),
                        "train_accuracy": float(train_metrics["accuracy"]),
                        "train_logloss": float(train_metrics["logloss"]),
                        "train_label_pos_rate": float(train_metrics["label_pos_rate"]),
                    }
                            mlflow.log_metrics(metrics_to_log, step=epoch
            print(
                f"Epoch {epoch + 1}/{tc.epochs} | train_loss={avg_train_loss:.4f} | "
                f"val_loss={avg_val_loss:.4f} | val_auc={val_metrics['auc_roc']:.4f} | "
                f"val_pr_auc={val_metrics['auc_pr']:.4f} | "
                f"val_prec={val_metrics['precision']:.4f} val_rec={val_metrics['recall']:.4f} "
                f"val_f1={val_metrics['f1']:.4f} | scale={_scale_val:.2f} | "
                f"val_nonfinite uemb={val_uemb_bad} cemb={val_cemb_bad} logits={val_logits_nonfinite}"
                        if train_metrics is not None:
                print(
                    f"           train_auc={train_metrics['auc_roc']:.4f} train_pr_auc={train_metrics['auc_pr']:.4f} "
                    f"train_prec={train_metrics['precision']:.4f} train_rec={train_metrics['recall']:.4f} "
                    f"train_f1={train_metrics['f1']:.4f} (sample_n={train_eval_kept})"
                            runlog.write(
                "EPOCH_DONE "
                f"epoch={epoch + 1}/{tc.epochs} "
                f"train_loss={avg_train_loss:.6f} "
                f"val_loss={avg_val_loss:.6f} "
                f"val_auc={val_metrics['auc_roc']:.6f} val_auc_pr={val_metrics['auc_pr']:.6f} "
                f"val_prec={val_metrics['precision']:.6f} val_rec={val_metrics['recall']:.6f} "
                f"val_f1={val_metrics['f1']:.6f} val_acc={val_metrics['accuracy']:.6f} "
                f"val_logloss={val_metrics['logloss']:.6f} "
                f"val_cm_tn={int(val_metrics['tn'])} val_cm_fp={int(val_metrics['fp'])} "
                f"val_cm_fn={int(val_metrics['fn'])} val_cm_tp={int(val_metrics['tp'])}"
            
        if best_snapshot is not None:
            best_user_state = best_snapshot["user_tower"]
            best_client_state = best_snapshot["client_tower"]
            ckpt_selection = "best_val_auc"
            mlflow.log_metrics({"best_val_auc": best_val_auc, "best_epoch": float(best_epoch)}, step=tc.epochs            print(
                f"Best epoch: {best_epoch + 1}/{tc.epochs} val_auc={best_val_auc:.4f} "
                "(weights exported to artifacts + MLflow)."
                    else:
            best_user_state = {k: v.detach().cpu().clone() for k, v in model.user_tower.state_dict().items()}
            best_client_state = {k: v.detach().cpu().clone() for k, v in model.client_tower.state_dict().items()}
            ckpt_selection = "last_epoch"
            print("No val_auc improvement recorded; exporting last-epoch towers."
        model.user_tower.load_state_dict({k: v.to(device) for k, v in best_user_state.items()}        model.client_tower.load_state_dict({k: v.to(device) for k, v in best_client_state.items()}        if best_snapshot is not None and "log_scale" in best_snapshot:
            model.log_scale.data.copy_(best_snapshot["log_scale"].to(device)
        _ex = next(iter(val_loader)        input_example = [
            _ex.user_cat.numpy(),
            _ex.user_num.numpy(),
            _ex.user_multi.numpy(),
            _ex.client_cat.numpy(),
            _ex.client_num.numpy(),
            _ex.client_multi.numpy(),
        ]
        try:
            mlflow.pytorch.log_model(
                model,
                name="model",
                input_example=input_example,
                serialization_format="pt2",
                    except Exception as e:
            print(f"MLflow pt2 export failed ({e!r}); falling back to pickle serialization."            mlflow.pytorch.log_model(model, name="model", input_example=input_example
        base = cfg.paths.artifacts_base
        user_tower_uri = artifact_uri(base, "artifacts", "user_tower", "user_tower_state.pt"        vocab_uri = artifact_uri(base, "artifacts", "vocab_artifact", "vocab_artifact.pkl"        client_emb_uri = artifact_uri(base, "artifacts", "client_embeddings", "client_embeddings.parquet"
        user_multi_vocab_sizes = [fa.user_multi_vocabs[c].size for c in fa.user_multi_cols]
        user_multi_emb_dims = [embedding_dim_for_cardinality(v) for v in user_multi_vocab_sizes]

        user_ckpt = {
            "state_dict": best_user_state,
            "emb_dim": tc.embed_dim,
            "user_vocab_sizes": user_vocab_sizes,
            "user_num_dim": user_num_dim,
            "user_multi_vocab_sizes": user_multi_vocab_sizes,
            "user_multi_emb_dims": user_multi_emb_dims,
            "user_multi_cols": fa.user_multi_cols,
            "multi_cat_max_tokens": tc.multi_max_tokens,
            "multi_cat_pool": tc.multi_cat_pool,
            "num_cross_layers": tc.dcn_cross_layers,
            "user_deep_hidden": list(tc.mlp_hidden_dims),
            "log_scale": float(model.log_scale.item()),
            "checkpoint_selection": ckpt_selection,
            "best_val_auc": float(best_val_auc) if best_snapshot is not None else None,
            "best_epoch_0based": int(best_epoch) if best_snapshot is not None else None,
            "use_pretrained_cat": use_pt,
            "pretrained_emb_dim": tc.pretrained_emb_dim,
            "target_cat_emb_dim": tc.pretrained_cat_emb_dim,
            "freeze_base": tc.freeze_pretrained_base,
        }
        _ubuf = io.BytesIO(        torch.save(user_ckpt, _ubuf        _write_bytes(user_tower_uri, _ubuf.getvalue()        print(f"Saved user tower to: {user_tower_uri} (selection={ckpt_selection})"
        vocab_artifact = {
            "user_vocabs": {k: vocab_to_dict(v) for k, v in fa.user_vocabs.items()},
            "user_multi_vocabs": {k: vocab_to_dict(v) for k, v in fa.user_multi_vocabs.items()},
            "user_cat_cols": fa.user_cat_cols,
            "user_num_cols": fa.user_num_cols,
            "user_multi_cols": fa.user_multi_cols,
            "device_id_col": fc.device_id_col,
            "multi_cat_max_tokens": tc.multi_max_tokens,
            "multi_cat_pool": tc.multi_cat_pool,
        }
        _vbuf = io.BytesIO(        pickle.dump(vocab_artifact, _vbuf, protocol=pickle.HIGHEST_PROTOCOL        _write_bytes(vocab_uri, _vbuf.getvalue()        print(f"Saved vocab artifact to: {vocab_uri}"
        client_id_col = _resolve_client_id_column(train_df, fc.client_id_col        client_df = train_df.drop_duplicates(subset=[client_id_col]).copy(        client_df["client_id"] = client_df[client_id_col]

        from two_tower.features.encode import encode_cats, encode_multi_matrix, encode_nums

        model.client_tower.eval(        _client_cat = encode_cats(client_df, fa.client_cat_cols, fa.client_vocabs        _client_num = encode_nums(client_df, fa.client_num_cols        _client_multi = encode_multi_matrix(
            client_df, fa.client_multi_cols, fa.client_multi_vocabs, tc.multi_max_tokens
                embs: list[np.ndarray] = []
        with torch.no_grad():
            for start in range(0, len(client_df), tc.batch_size):
                end = start + tc.batch_size
                cc = _client_cat[start:end].to(device                cn = _client_num[start:end].to(device                cm = _client_multi[start:end].to(device                emb = model.client_tower(cc, cn, cm).cpu().numpy().astype("float32"                embs.append(emb        client_emb_matrix = np.concatenate(embs, axis=0        out_df = pd.DataFrame({"client_id": client_df["client_id"].values}        out_df["embedding"] = [row.tolist() for row in client_emb_matrix]
        out_df.to_parquet(client_emb_uri, index=False        print(f"Wrote client embeddings to: {client_emb_uri}"
        print("[train_and_log] Done."        runlog.write(f"FINISH ok=true elapsed_s={time.time() - t_start:.1f}"    except Exception as e:
        runlog.write(f"FINISH ok=false elapsed_s={time.time() - t_start:.1f} error={e!r}"        raise


## Parse config
Converts `TRAIN_CONFIG_YAML` into a `PipelineConfig` object. Re-run this cell after editing the YAML above.


In [ ]:
pipeline_cfg = load_pipeline_config_from_yaml_string(TRAIN_CONFIG_YAML)
print("train path :", pipeline_cfg.paths.train)
print("label_col  :", pipeline_cfg.features.label_col)
print("epochs     :", pipeline_cfg.train.epochs)


## Optional: per-client downsampling
Set `DOWNSAMPLE_TRAIN = True` to balance negatives:positives within each client
and (optionally) equalise row counts across clients before training.
Validation data is never modified unless `DOWNSAMPLE_VAL = True`.


In [ ]:
# ── Per-client downsampling (optional) ────────────────────────────────────
# Set DOWNSAMPLE_TRAIN = True to balance negatives:positives per client.
# Set TRAIN_EQUALIZE_CLIENT_ROWS = True to cap every client to the same row count.
# Validation is never downsampled unless DOWNSAMPLE_VAL = True.

DOWNSAMPLE_TRAIN = False
TRAIN_NEG_PER_POS = 10
TRAIN_EQUALIZE_CLIENT_ROWS = True
DOWNSAMPLE_VAL = False
DOWNSAMPLE_RANDOM_STATE = 42


def _balance_train_per_client(df, neg_per_pos, ycol, ccol, rs=42):
    """Undersample within each client so negatives:positives ≈ neg_per_pos:1."""
    if neg_per_pos is None:
        return df
    r = max(1, int(neg_per_pos))
    parts = []
    for i, (_cid, g) in enumerate(df.groupby(ccol, sort=False)):
        pos = g[g[ycol] == 1]
        neg = g[g[ycol] == 0]
        n_p, n_n = len(pos), len(neg)
        if n_p == 0 or n_n == 0:
            parts.append(g)
            continue
        k = min(n_p, n_n // r)
        rs_p, rs_n = rs + 2 * i, rs + 2 * i + 1
        if k == 0:
            m = min(n_p, n_n)
            parts.append(pd.concat(
                [pos.sample(n=m, random_state=rs_p), neg.sample(n=m, random_state=rs_n)],
                ignore_index=True,
            ))
        else:
            parts.append(pd.concat(
                [pos.sample(n=k, random_state=rs_p), neg.sample(n=k * r, random_state=rs_n)],
                ignore_index=True,
            ))
    return pd.concat(parts, ignore_index=True) if parts else df.iloc[:0]


def _equalize_clients_to_min_rows(df, neg_per_pos, ycol, ccol, rs=123):
    """Downsample each client to the same row count while keeping neg:pos ratio."""
    if neg_per_pos is None:
        return df
    r = max(1, int(neg_per_pos))
    stats = [
        (cid, int((g[ycol] == 1).sum()), int((g[ycol] == 0).sum()))
        for cid, g in df.groupby(ccol, sort=False)
    ]
    if not stats:
        return df
    m = min(min(pos_c, (neg_c // r) if r else pos_c) for _, pos_c, neg_c in stats)
    if m <= 0:
        return df
    parts = []
    for i, (_, g) in enumerate(df.groupby(ccol, sort=False)):
        pos = g[g[ycol] == 1]
        neg = g[g[ycol] == 0]
        rs_p, rs_n = rs + 3 * i, rs + 3 * i + 1
        parts.append(pd.concat(
            [pos.sample(n=m, random_state=rs_p), neg.sample(n=m * r, random_state=rs_n)],
            ignore_index=True,
        ))
    return pd.concat(parts, ignore_index=True) if parts else df.iloc[:0]


## Load data → prepare features → train
This is the only cell you need to re-run if you want to retrain after
editing the config or downsampling toggles above.


In [ ]:
train_df, val_df = load_train_validation_frames(pipeline_cfg)
print(f"train shape: {train_df.shape}  |  val shape: {val_df.shape}")

ycol = pipeline_cfg.features.label_col
ccol = pipeline_cfg.features.client_id_col

if DOWNSAMPLE_TRAIN:
    train_df = _balance_train_per_client(
        train_df, TRAIN_NEG_PER_POS, ycol, ccol, rs=DOWNSAMPLE_RANDOM_STATE
    )
    if TRAIN_EQUALIZE_CLIENT_ROWS:
        train_df = _equalize_clients_to_min_rows(
            train_df, TRAIN_NEG_PER_POS, ycol, ccol, rs=DOWNSAMPLE_RANDOM_STATE + 1000
        )
    train_df = train_df.sample(frac=1.0, random_state=DOWNSAMPLE_RANDOM_STATE).reset_index(drop=True)
    print(f"After downsampling — train: {train_df.shape}")

if DOWNSAMPLE_VAL:
    val_df = _balance_train_per_client(
        val_df, TRAIN_NEG_PER_POS, ycol, ccol, rs=DOWNSAMPLE_RANDOM_STATE + 7
    )
    if TRAIN_EQUALIZE_CLIENT_ROWS:
        val_df = _equalize_clients_to_min_rows(
            val_df, TRAIN_NEG_PER_POS, ycol, ccol, rs=DOWNSAMPLE_RANDOM_STATE + 2000
        )
    val_df = val_df.sample(frac=1.0, random_state=DOWNSAMPLE_RANDOM_STATE + 7).reset_index(drop=True)
    print(f"After downsampling — val: {val_df.shape}")

feature_artifacts = prepare_training_features(train_df, val_df, pipeline_cfg)

from dataclasses import replace
pipeline_cfg = replace(
    pipeline_cfg,
    train=replace(pipeline_cfg.train, train_eval_max_examples=200_000),
)

train_and_log(
    cfg=pipeline_cfg,
    train_df=train_df,
    val_df=val_df,
    feature_artifacts=feature_artifacts,
)
